# Where Will Floods Hit Next? – Flood Extent of Pakistan by Tehsil (2022–2025)
**Dataset:** "A Multi-Resolution Geospatial Dataset of Monsoon Flood Extent and Spatial Clustering for Pakistan (2022–2025)" (Zenodo, CC BY 4.0).
Flood area was detected by the VIIRS satellite and combined with UNOSAT flood maps, for **160 districts** and **577 tehsils**, for each monsoon from 2022 to 2025.

**Goal:** Using the flood situation of **this year**, predict which tehsils will be **seriously flooded next year** (more than 10% of their area under water).
This is a flood-susceptibility style task like Papers 2 to 5, but the dataset has **no rain, height or slope columns**, only past flood extent.

**Models:** a simple baseline ("same as last year"), Logistic Regression, Random Forest and XGBoost

## Step 1: Load the libraries and the data

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

# Find the CSV files automatically inside /kaggle/input
all_csv = glob.glob('/kaggle/input/**/*.csv', recursive=True)
def read(key):
    path = [f for f in all_csv if key in f.lower()][0]
    print('Using file:', path)
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp1252')

tehsil = read('tehsil_level_data')
district = read('district_level_data')
print('Tehsil table:', tehsil.shape, '| District table:', district.shape)
tehsil.head()

## Step 2: Look at the data
Each row is **one tehsil in one year**. The main columns are:
- `flooded_km2` and `pct_flooded`: how much area (km² and %) was flooded
- `gi_z` and `hotspot`: a statistical test that finds **clusters (hot spots)** of heavily flooded neighbouring areas
- `lisa_quad`: another cluster test ("High-High" = a heavily flooded tehsil surrounded by heavily flooded tehsils)

In [ ]:
# Flooded area (km²) of each province in each year
by_prov = tehsil.pivot_table(index='province_name', columns='year', values='flooded_km2', aggfunc='sum').round(0)
display(by_prov)
by_prov.T.plot(kind='bar', stacked=True, figsize=(9, 4.5))
plt.ylabel('Flooded area (km²)'); plt.title('Flooded area by province and year'); plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left'); plt.tight_layout(); plt.show()

print('Share of the total flooded area (%) – which province was hit the most each year?')
display((by_prov / by_prov.sum() * 100).round(0))

In [ ]:
# Summary of each year
summary = tehsil.groupby('year').agg(
    tehsils=('tehsil_code', 'count'),
    total_flooded_km2=('flooded_km2', 'sum'),
    average_pct_flooded=('pct_flooded', 'mean'),
    share_of_tehsils_over_10pct=('pct_flooded', lambda s: (s > 10).mean() * 100)).round(1)
display(summary)

print('Hot spot tehsils in each year:')
display(tehsil.groupby('year')['hotspot'].value_counts().unstack().fillna(0).astype(int))

In [ ]:
# Does the flood of one year look like the flood of another year?
# 1 = the same tehsils are flooded, 0 = no relation
same = tehsil.pivot(index='tehsil_code', columns='year', values='pct_flooded').corr().round(2)
display(same)

## Step 3: What exactly do we predict?
- **Inputs (features):** the flood situation of tehsil T in year **N** (% flooded, km² flooded, tehsil area, hot-spot scores, the average flooding of its district, and its province).
- **Output (answer):** will the same tehsil be **seriously flooded (more than 10% of its area) in year N+1**? 1 = yes, 0 = no.
- **Training data:** pairs 2022→2023 and 2023→2024. **Test data:** pairs 2024→2025 (the models never see 2025 while learning).
- **No leakage:** the inputs always come from the earlier year, and the answer from the next year.

In [ ]:
THRESHOLD = 10   # "seriously flooded" = more than 10% of the tehsil area

data = tehsil.copy()
data['hot'] = data['hotspot'].str.startswith('Hot').astype(int)            # 1 = hot spot
data['high_high'] = (data['lisa_quad'] == 'High-High').astype(int)         # 1 = flooded and surrounded by flooded areas
data['district_mean'] = data.groupby(['province_name', 'district_name', 'year'])['pct_flooded'].transform('mean')
prov = pd.get_dummies(data['province_name'], prefix='prov').astype(int)
data = pd.concat([data, prov], axis=1)
features = ['pct_flooded', 'flooded_km2', 'area_km2', 'gi_z', 'hot', 'high_high', 'district_mean'] + list(prov.columns)

# Put this year's row next to the same tehsil's flood of the NEXT year
nxt = data[['tehsil_code', 'year', 'pct_flooded']].copy()
nxt['year'] -= 1
nxt = nxt.rename(columns={'pct_flooded': 'next_pct'})
pairs = data.merge(nxt, on=['tehsil_code', 'year'])
pairs['y'] = (pairs['next_pct'] > THRESHOLD).astype(int)

train, test = pairs[pairs['year'] <= 2023], pairs[pairs['year'] == 2024]
X_train, y_train, X_test, y_test = train[features], train['y'], test[features], test['y']
print('Training rows:', len(train), '| seriously flooded next year:', round(y_train.mean() * 100, 1), '%')
print('Test rows    :', len(test), '| seriously flooded in 2025  :', round(y_test.mean() * 100, 1), '%  (', int(y_test.sum()), 'tehsils )')

## Step 4: The models
| Model | Idea |
|---|---|
| Baseline | "A tehsil that was seriously flooded this year will be flooded again next year." The simplest guess. |
| Logistic Regression | Draws one straight line between "flooded again" and "not flooded again" |
| Random Forest | 300 decision trees vote |
| XGBoost | Trees built one after another; each fixes the mistakes of the earlier ones |

Only about 1 tehsil in 10 is seriously flooded, so we tell the models to **pay extra attention to the flooded tehsils** (class weights).

In [ ]:
def evaluate(name, y_true, pred, prob):
    return {'Model': name, 'Accuracy': accuracy_score(y_true, pred), 'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred), 'F1-score': f1_score(y_true, pred), 'ROC-AUC': roc_auc_score(y_true, prob)}

results, predictions = [], {}

# 0) Baseline: flooded this year (more than 10%) -> will be flooded again
pred = (test['pct_flooded'] > THRESHOLD).astype(int).values
prob = test['pct_flooded'].values
results.append(evaluate('Baseline (same as last year)', y_test, pred, prob)); predictions['Baseline'] = (pred, prob)

# 1) Logistic Regression
lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000)).fit(X_train, y_train)
prob = lr.predict_proba(X_test)[:, 1]; pred = (prob >= 0.5).astype(int)
results.append(evaluate('Logistic Regression', y_test, pred, prob)); predictions['Logistic Regression'] = (pred, prob)

# 2) Random Forest
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', min_samples_leaf=3, random_state=42, n_jobs=-1).fit(X_train, y_train)
prob = rf.predict_proba(X_test)[:, 1]; pred = (prob >= 0.5).astype(int)
results.append(evaluate('Random Forest', y_test, pred, prob)); predictions['Random Forest'] = (pred, prob)

# 3) XGBoost
spw = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=spw, eval_metric='logloss', random_state=42).fit(X_train, y_train)
prob = xgb.predict_proba(X_test)[:, 1]; pred = (prob >= 0.5).astype(int)
results.append(evaluate('XGBoost', y_test, pred, prob)); predictions['XGBoost'] = (pred, prob)
print('Done: 4 models')

## Step 5: Compare the results

In [ ]:
res = pd.DataFrame(results).set_index('Model').round(3)
display(res)
res[['Precision', 'Recall', 'F1-score', 'ROC-AUC']].plot(kind='bar', figsize=(10, 4), title='Model comparison on 2025 (test data)')
plt.ylabel('Score (higher is better)'); plt.xticks(rotation=15); plt.show()
print('Best model by F1-score:', res['F1-score'].idxmax())

In [ ]:
# Confusion matrix: how many tehsils each model got right or wrong
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, (pred, prob)) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['Not flooded', 'Flooded']).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# ROC curve: closer to the top-left corner = better model
fig, ax = plt.subplots(figsize=(6, 5))
for name, (pred, prob) in predictions.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random guess'); ax.legend(); ax.set_title('ROC curves'); plt.show()

## Step 6: Which inputs did XGBoost use the most?

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=features).sort_values()
imp.plot(kind='barh', figsize=(7, 5), color='tab:green', title='XGBoost feature importance')
plt.show()

## Step 7: Reality check by province (2025)
In 2025 the flood pattern changed. Which provinces did the models get right, and where did they fail?

In [ ]:
chk = pd.DataFrame({'province': test['province_name'].values, 'really_flooded': y_test.values,
                    'baseline_flagged': predictions['Baseline'][0], 'xgboost_flagged': predictions['XGBoost'][0]})
chk['xgboost_correct'] = chk['really_flooded'] & chk['xgboost_flagged']
out = chk.groupby('province').agg(really_flooded=('really_flooded', 'sum'), baseline_flagged=('baseline_flagged', 'sum'),
                                  xgboost_flagged=('xgboost_flagged', 'sum'), xgboost_correct=('xgboost_correct', 'sum'))
display(out[out.sum(axis=1) > 0])

## Step 8: Why we split by time (an unfair test for comparison)
If we mix all years and split **randomly**, the same tehsil appears in both training and test (in different years), so the model can "recognise" it.
The score looks better, but it is not a real forecast.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(pairs[features], pairs['y'], test_size=0.3, random_state=42, stratify=pairs['y'])
m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(), eval_metric='logloss', random_state=42).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:, 1]
print('XGBoost with a RANDOM split (unfair):  ROC-AUC', round(roc_auc_score(yte, p), 3), '| F1', round(f1_score(yte, (p >= 0.5).astype(int)), 3))
print('XGBoost with a TIME split (honest)  :  ROC-AUC', res.loc['XGBoost', 'ROC-AUC'], '| F1', res.loc['XGBoost', 'F1-score'])

## Conclusion
- We used the flood extent of 577 tehsils (2022–2025) to predict which tehsils will be seriously flooded **next year**.
- Floods of 2022, 2023 and 2024 look very similar (Step 2), so "same as last year" is already a strong guess, and the models only do about as well as the baseline.
- **2025 was different:** the flooding moved towards Punjab and away from Sindh (Step 2 and Step 7), so models that learned from the old pattern gave many false alarms in Sindh.
- A random split would make the models look better than they really are (Step 8), so we used a time-based split.
- **Limitations:** only 4 years, no rain / height / slope / river data, and the flood map is a lower estimate (optical satellites cannot see through clouds).
- **Link to my thesis:** past flood maps alone cannot tell where the **next** flood will hit. My thesis adds **satellite rainfall, weather and terrain data** to predict flash floods **hours ahead** in **KP and Balochistan**.